# Introduction

This notebook is part of the bachelor's thesis Solving the 3-SAT problem using quantum algorithms done by Arnau Casau at FIB - UPC Universitat Politècnica de Catalunya.

This code is a continuation of the previous notebook called *Part-1-3SAT-Quantum-Computing.ipynb*, in which we can find an implementation of Grover's algorithm to solve instances of the 3-SAT problem. The previous algorithm allows us to achieve a quadratic speedup over a random guessing classical algorithm.

In the following paragraphs, we will see that there are classical algorithms that can improve what we've got with Grover's and what we can do with a quantum computer to obtain an even better algorithm than the classical ones.

# Schöning's algorithm

Schöning's algorithm is a classical algorithm that allows us to find satisfiable assignments to 3-SAT instances. This algorithm is quite simple, and here we can see what it looks like:

```
input: a formula in k-CNF whit n variables
Guess and initial assignment x ∈ {0,1}^n
Repeat 3n times:
    If the formula is satisfied by the actual assignment:
        Stop and accept
    Let C be some clause not being satisfied by the actual assignment
    Pick one of the <= k literals in the clause at random and flip its value in the current assignment
```
This algorithm is polynomial to the number of variables in our formula, and it has a probability of success of at least $(2-{2\over{k}})^{-n}$. In this case, because we are trying to solve 3-SAT, we have a probability of $({4\over{3}})^{-n}$. To find a satisfiable assignment with high probability, we will have to repeat the algorithm ${1\over{({4\over{3}})^{-n}}}=({4\over{3}})^{n}$ times. If we do so, we see that the time complexity after the repetitions is $O(({4\over{3}})^{n}\cdot poly(n))$, which is better than the one achieved with Grover's.

# Amplitude Amplification and Schöning's algorithm

Even though Schöning's algorithm is better than Grover's algorithm of the previous notebook, we still can do something about it. If we create a circuit that carries out the bit-toggling part of Schöning's algorithm, we can use this as the oracle of Grover's algorithm (generalized as Amplitude Amplification) and achieve a quadratic speedup over the number of repetitions that we should have done with classical Schöning's algorithm. This leads us to a time complexity of $O(\sqrt{({4\over{3}})^{n}}\cdot poly(n))$.

Schöning's algorithm is a random walk, but we can create a circuit, in which we view the 3n step as a deterministic algorithm with an additional input r ∈ {0,1}$^n$ x {1,2,3}$^{3n}$, where the first n bits determine x, and the last 3n entries determine which variable of the leftmost false clauses will be flipped in the 3n steps of the random walk (In the algorithm proposed below, we will always use the first unsatisfied clause to simplify the circuit). Finally, we will use the Amplitude Amplification on the space of all such r.

Here we can find a table with all the algorithm that I've mentioned above and their time complexity:


| Algorithm  | Type | Time complexity |
| -------------: |:--:| :------------- |
| Random guessing | Classical | $O(2^n)$  |
| Grover search |Quantum | $O(1.4142^n)$  |
| Schöning | Classical| $O(1.3334^n)$  |
| Amplitude Amplification/Schöning |Quantum/Classical | $O(1.1547^n)$  |


In [5]:
%pip install qiskit

# imports
import math

# importing Qiskit
from qiskit import QuantumCircuit, transpile
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.quantum_info import Statevector
from qiskit.qasm2 import dump
from qiskit.circuit import Qubit
from functools import wraps
from qiskit import QuantumCircuit as _BaseCircuit
import io
import subprocess
from qiskit.transpiler.passes import RemoveBarriers
import numpy as np
import tempfile


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Like in the previous notebook, we can read the formula from a text file. The only difference, in this case, is the number of total qubits we need to create the circuit. We need n qubits to represent all the possible assignments, where n is the number of variables in the formula.

We also need, for each of the 3n iterations, ceil(log(clause)) qubits and two qubits more that we will use to represent the random choice the algorithm does when it flips the value of one variable in the leftmost false clause. Finally, we will need a counter with ceil(log(clauses)) extra qubits to determine the first unsatisfied clause for an assignment and one extra qubit to represent the state |-⟩ to perform a phase shift of -1 to all solutions in the oracle.

This expression summarizes all that has been said above, where n is the number of variables in the formula and c is the number of clauses thereof:

Number of qubits: $n + 3n \cdot(2 + \log c+1) + \log c+1 + 1$

In [6]:
#f = open('CBS_k3_n100_m403_b10/CBS_k3_n100_m403_b10_10.cnf')
#f = open('uf20-91/uf20-01.cnf')
f = open('uf50-218/uf50-01.cnf')

f.readline()
f.readline()
f.readline()

line = f.readline()
while line[0] != 'p':
    line = f.readline()
print(line)
info = line.split()
n_var = int(info[2])
n_clause = int(info[3])

def bits_needed(n):
    log = 0
    while (n > 0):
        n = n // 2
        log += 1
    return log

log_n_clause = bits_needed(n_clause)

lines = [line.rstrip('\n') for line in f]

formula = []
for i in lines:
    l = i.split()
    formula.append([int(l[0]),int(l[1]),int(l[2])])

# Don't need n_clause*n_var qubits at each Schoning step, just need log_n_clause) as given by https://arxiv.org/pdf/2404.15235
n_qubits = n_var + 3*n_var*(2 + log_n_clause+1) + log_n_clause+1 + 1
n_measures = 7*n_var
iters = 3*n_var
n_qubits

p cnf 50  218 



1710

The first step of the Amplitude Amplification is to initialize all the qubits. As we have seen above, the state space of the algorithm will be all the inputs r ∈ {0,1}$^n$ x {1,2,3}$^{3n}$. To create this state is easy to see that the first n qubits have to be in a superposition because we want to represent all the possible assignments of our formula. To achieve that, we only have to use a Hadamard gate in those qubits as we have done in the previous notebook.

To represent the choice of one literal in each of the 3n steps in Schöning's algorithm, we can create 3n times the state $1\over{\sqrt{3}}$ $(|00⟩ + |01⟩ + |10⟩)$, which is a superposition state and will collapse with equal probability in the states |00⟩, |01⟩ o |10⟩. We can use the decimal representation of these states to decide which literal in the chosen false clause will be modified.

In order to build the state $1\over{\sqrt{3}}$ $(|00⟩ + |01⟩ + |10⟩)$ we will use 2 qubits initialized to |00⟩ and two gates. The first gate we need is Ry($\theta$), which will perform a rotation of $\theta$ degrees around the y-axis. The second one we need is the Hadamard gate but his controlled version, as we will see as follows.

The angle $\theta$ we will need to create the state is $\theta = 2 \cdot arccos \cdot (\sqrt{2\over3})$.

|00⟩ -> Perform the Ry($\theta$) gate on the first qubit -> $(\sqrt{{2\over3}}|0⟩ + \sqrt{{1\over3}}|1⟩) |0⟩ = \sqrt{{2\over3}}|00⟩ + \sqrt{{1\over3}}|10⟩$ -> Use the Hadamard gate on the second qubit when the first one is 0 -> $\sqrt{{2\over3}}|0⟩ (\sqrt{{1\over2}}|0⟩ + \sqrt{{1\over2}}|1⟩) + \sqrt{{1\over3}}|10⟩ = \sqrt{{1\over3}}|00⟩ + \sqrt{{1\over3}}|01⟩ + \sqrt{{1\over3}}|10⟩ = {1\over{\sqrt{3}}} (|00⟩ + |01⟩ + |10⟩)$

To better understand why the angle $\theta$ has the value said above, we need to remember the matrix that represents the gate:

$$
Ry(\theta) = \left(\begin{array}{cc} 
cos({\theta\over{2}}) & -sin({\theta\over{2}})\\
sin({\theta\over{2}}) & cos({\theta\over{2}})
\end{array}\right)
$$

In the following piece of code, we can find the initialization of all qubits doing the above said. The last qubit is initialized to the state |-⟩ using a NOT gate and a Hadamard gate, as we have seen in the previous notebook.

The circuit is encapsulated into a gate called *Initialization* that we will use later.

In [7]:
def initialization():
    qc = QuantumCircuit(n_qubits)
    for i in range(n_var):
        qc.h(i)

    ini = n_var+log_n_clause+1
    theta = 2*math.acos(math.sqrt(2/3))
    for i in range(iters):
        qc.u(theta,0,0,ini)
        qc.x(ini)
        qc.ch(ini, ini+1)
        qc.x(ini)
        ini += 2+log_n_clause+1

    qc.x(n_qubits-1)
    qc.h(n_qubits-1)

    initial = qc.to_gate()
    initial.name = "initialization"
    return initial
    # qc.draw()

The next step in the Amplitude Amplification is to design the oracle to mark the solutions of our formula, but this is a large circuit because we have to simulate the 3n iterations of Schöning's algorithm. To better understand all the parts, we can take as an example only the first iteration of Schöning's in a small formula of four variables and two clauses:

(-x1 v -x2 v x3) ∧ (x2 v x3 v -x4)

The three phases are:

* (Green) The initialization of all qubits we have done above.
* (Orange) Check the clauses to determine which one has been satisfied by the actual assignment using auxiliary qubits.
* (Red) Flip one out of three literals of the leftmost unsatisfied clause.

As we can see in the orange step, to determine whether a clause has been satisfied, we can do the same as in the previous notebook. We can use a Toffoli gate with the qubits representing the negation of the clauses' literals as control. Because we negate their values, we know that if the auxiliary qubit ends up being |1⟩, that means that the clause wasn't satisfied because the control qubits are a conjunction of the only values that make each literal false.

On the other hand, in the red step, we can see we are also using Toffoli gates to flip the values of one of the literals in the first unsatisfied clause. In this step, we will have as many Toffoli gates as the number of clauses multiplied by three because we will use a Toffoli gate for each literal in each of its appearances. The control qubits are chosen in order to make sure that the state $1\over{\sqrt{3}}$ $(|00⟩ + |01⟩ + |10⟩)$ will collapse, in that iteration, in the same position of the literal affected by the Toffoli gate. We also control that the clause that the literal belongs to hasn't been satisfied, and all the previous ones have been satisfied (the value of the counter should be 0) because we only want to flip one variable at a time.

Once we have simulated all the 3n iterations of Schöning's algorithm, we again have to check whether the assignment satisfies the formula. In the affirmative case, we have to perform a phase shift of -1 to mark the assignment as a solution. This step is the same as we have done in Grover's algorithm in the previous notebook. In the following code, we can see the orange step, the red step, and the last checking of the clauses before marking the state if it's a solution (We can see it as another orange step using the last auxiliary qubits). As before, we encapsulate the circuit in a gate called *Schoning* to use it in the next cell to complete the oracle.

In [8]:
# New controlled increment function
def controlled_increment(circ: QuantumCircuit, ctrl: list[int], target: list[int]):
    """Add 1 mod 2^n to `target` iff all `ctrl` is 1, using no ancilla.
       - target: list of n qubit‐indices (LSB first)
       - uses mcx(..., mode='noancilla') under the hood"""
    # ripple‐carry using only no-ancilla mcx
    n = len(target)
    for i in range(n-1, -1, -1):
        controls = ctrl + target[:i]
        circ.mcx(controls, target[i], mode='noancilla')

In [9]:
# Return a quantum circuit corresponding to a single step of 3n steps in Schoning's algorithm
def schoning_step(first_clause, first_bit=None):
    qc = QuantumCircuit(n_qubits)
    for clause in formula:
            # Evaluate clause to check if unsatisfied and write result to qubit at first_clause
            control = []
            state = ''
            for i in range(len(clause)):
                lit = clause[i]
                lit_aux = abs(lit)-1
                if lit < 0:
                    state+= '1'
                else:
                    state+= '0'
                control.append(lit_aux)
            
            counter_bits = list(range(first_clause+1, first_clause+1+log_n_clause))
            counter_state = '0'*len(counter_bits)
            qc.mcx(control + counter_bits, first_clause, ctrl_state=counter_state+state[::-1])

            if first_bit is not None:
                # If counter is 0 and the clause is unsatisfied, flip the literal given by the randomness starting at first_bit
                control = [first_clause] + counter_bits + [first_bit+1, first_bit]
            
                lit1 = abs(clause[0])-1
                lit2 = abs(clause[1])-1
                lit3 = abs(clause[2])-1

                qc.mcx(control,lit1, ctrl_state='00'+counter_state+'1')
                qc.mcx(control,lit2, ctrl_state='01'+counter_state+'1')
                qc.mcx(control,lit3, ctrl_state='10'+counter_state+'1')

            # Increment if we have found an unsatisfied clause.
            controlled_increment(qc, ctrl=[first_clause], target=counter_bits)

def schoning_alg():
    qc = QuantumCircuit(n_qubits)
    for it in range(iters):
            first_bit = n_var+(2+log_n_clause+1)*it+log_n_clause+1
            first_clause = n_var+(2+log_n_clause+1)*it
            schoning_step(first_clause, first_bit)

    # Final check to see if all clauses are satisfied
    schoning_step(n_qubits-1-log_n_clause)

    # Note that if n_qubits-1-log_n_clause is 0, then all clauses are satisfied
    schoning_gate = qc.to_gate()
    schoning_gate.name = "schoning_alg"
    return schoning_gate
# qc.draw(fold=10000)

In the next cell, we can finish our oracle. The oracle code is divided into two cells because, thanks to the gate with the circuit above, we can quickly generate its complex conjugate transpose with the Qiskit function *inverse*. We need to do so to revert all the computations before going to the next step of the Amplitude Amplification.

Between the gate called *Schoning* and its inverse, we need to use a Toffoli gate to negate the last qubit performing a phase shift of -1 and mark the solution states. In that Toffoli gate, we will use the negation of the auxiliary qubits modified in the last step in the previous cell because when they are |0⟩, it means the clause is satisfied.

These three steps are encapsulated in another gated called *Oracle* which we will use in the Amplitude Amplification at the end.

In [10]:
def oracle():
    qc = QuantumCircuit(n_qubits)

    schoning_gate = schoning_alg()

    qc.append(schoning_gate, list(range(n_qubits)))

    control = n_qubits-1-log_n_clause-1
    qc.cx(control,n_qubits-1, ctrl_state='0')

    qc.append(schoning_gate.inverse(), list(range(n_qubits)))

    oracle = qc.to_gate()
    oracle.name = "oracle"
    return oracle
#qc.draw()

The next step is to define the generalized version of what we called in the previous notebook as steps 3, 4, and 5 of Grover's algorithm (inversion about the mean).

We used the Dirac notation to defined the step as $D = H^{⊗n}$ ($2|0⟩⟨0|^{⊗n} - I$) $H^{⊗n}$ = $2|\psi⟩⟨\psi| - I$.

In this case, we cannot use the same definition because it's not true anymore. It is not true that $H^{⊗n} |0⟩^{⊗n} =|\psi⟩$ because $|\psi⟩$ represents our initial state before running the oracle, and in our initialization, we use more gates than only Hadamard. Recall: We have created the state $1\over{\sqrt{3}}$ $(|00⟩ + |01⟩ + |10⟩)$.

A generalization of D, in Dirac notation, could be $D = A$ ($2|0⟩⟨0|^{⊗n} - I$) $A^†$ = $2|\psi⟩⟨\psi| - I$, where $A |0⟩^{⊗n} = |\psi⟩$.

Following this, it's clear that we have to apply the initial transformation where the transformation $A$ appears in the generalization. We also can see that we will use its complex conjugate transpose in place of $A^†$. 

To simplify the circuit, we will use the opposite of step 4, which is to perform a phase shift of -1 to the state $|0⟩^{⊗n}$. This simplification is equivalent because we are applying a global phase.


There is a change between the theoretical circuit and the following implementation to be able to run this circuit in a classical computer. The only Toffoli gate needed in step 4 has been divided into little Toffoli gates using the previous auxiliary qubits used in the oracle. This change is not necessary for the theoretical circuit, but it is to use and speed up the time needed by simulators.

In [11]:
l = list()
def diffuse_step():
    global l
    qc = QuantumCircuit(n_qubits)

    l = list()
    aux = list()

    ini = n_var
    for i in range(iters):
        for j in range(log_n_clause+1):
            aux.append(ini+j)
        ini += log_n_clause+1+2

    for j in range(log_n_clause+1):
        aux.append(ini+j)


    for i in range(n_var):
        qc.h(i)
        qc.x(i)
        l.append(i)

    ini = n_var+log_n_clause+1
    theta = 2*math.acos(math.sqrt(2/3))
    for i in range(iters):
        qc.x(ini)
        qc.ch(ini, ini+1)
        qc.x(ini)
        qc.u(-theta,0,0,ini)

        qc.x(ini)
        qc.x(ini+1)
        l.append(ini)
        l.append(ini+1)
        ini += 2+log_n_clause+1

    #qc.mct(l,n_qubits-1)
    original = l
    total = len(l)
    l2 = list()
    i = 0
    while total != 1:
        for j in range(math.ceil(total/4)):
            qc.mcx(l[4*j:4*j+4],aux[i])
            l2.append(aux[i])
            i+=1

        l = l2
        total = len(l)
        l2 = list()

    diff_part1 = qc.to_gate()
    diff_part1.name = "diffuse_step"
    return diff_part1
#qc.draw()

To finish with the generalization of the steps 3, 4, and 5 of Grover's algorithm or inversion about the mean, we need to use the above code encapsulated into a gate called *diff_part1*, negate the last qubit to perform the phase shift of step 4 and finally, use the gate *diff_part1* dagger (complex conjugate transpose) to reverse all the changes in the auxiliary qubits.

In the next cell, we can see the code:

In [12]:
def diffuser():
    qc = QuantumCircuit(n_qubits)

    diffuse_gate = diffuse_step()

    qc.append(diffuse_gate, list(range(n_qubits)))

    qc.mcx(l,n_qubits-1)

    qc.append(diffuse_gate.inverse(), list(range(n_qubits)))

    diffuser = qc.to_gate()
    diffuser.name = "Diffuser"
    return diffuser
#qc.draw()

At this point, we have all the ingredients to run the Amplitude Amplification simulating Schöning's algorithm as the oracle. As in the previous notebook, we can calculate an upper bound of the number of repetitions of our generalized Grover's operator as follows:

$R \le \lceil {{\pi}\over{4}} \cdot \sqrt{({4\over3})^n}  \rceil$

In [13]:
rounds = math.pi/4.0*math.sqrt((4/3)**n_var)
print("Result:",rounds)
print("Iterations R ≤",math.ceil(rounds))

if math.floor(rounds) == 0:
    rounds = round(rounds)
else:
    rounds = math.floor(rounds)

print("Iterations chosen:", rounds)

Result: 1043.6582165135815
Iterations R ≤ 1044
Iterations chosen: 1043


Finally, we can build the Amplitude Amplification's circuit:

In [18]:
import warnings
warnings.filterwarnings("ignore")

qc = QuantumCircuit(n_qubits,n_measures)

qc.append(initialization(), list(range(n_qubits)))
oracle_circuit = oracle()
diffuser_circuit = diffuser()
print('got oracle and diffuser')
for i in range(rounds):
        qc.append(oracle_circuit, list(range(n_qubits)))
        qc.append(diffuser_circuit, list(range(n_qubits)))

actual = 0

for i in range(n_var):
    qc.measure([i],[i])
    actual += 1

ini = n_var+log_n_clause+1
for i in range(iters):
    qc.measure([ini],[actual])
    qc.measure([ini+1],[actual+1])
    ini += 2+log_n_clause+1
    actual += 2

#qc.draw()

got oracle and diffuser


In [19]:
# Grover + Schoning full circuit
from qiskit.qasm2 import dump

with open(f'grover_3sat_schoning_{n_qubits}.qasm', 'w') as f:
    dump(qc, f)